# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

frame = con.sql("""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS early_impressions,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS early_clicks,
  SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') * 1.0
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15'), 0) AS early_ctr,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS early_avg_position,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS early_active_days,
  SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15') AS late_impressions
FROM march
WHERE gsc_data_available IS TRUE
  AND hash(content_hash_id) % 50 = 0
GROUP BY 1, 2
""").df()

df = frame[frame["early_impressions"] >= 20].copy()
df["late_impressions"] = df["late_impressions"].fillna(0)
df["label_declining"] = ((df["late_impressions"] / 16) < 0.8 * (df["early_impressions"] / 15)).astype(int)
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
print(df.shape, "declining rate:", round(df["label_declining"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2191, 9) declining rate: 0.329


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
features = ["early_impressions", "early_clicks", "early_ctr", "early_avg_position", "early_active_days"]
print("missing values per feature before filling:")
print(df[features].isna().sum())
X = df[features].fillna(0)          # all five are numeric; a missing early value means no activity to count
y = df["label_declining"]
groups = df["client_hash_id"]
print("feature matrix:", X.shape)
X.describe().round(3).T[["count", "mean", "min", "max"]]

missing values per feature before filling:
early_impressions     0
early_clicks          0
early_ctr             0
early_avg_position    0
early_active_days     0
dtype: int64
feature matrix: (2191, 5)


,count,mean,min,max
early_impressions,2191.0,1209.389,20.000,86860.000
early_clicks,2191.0,3.424,0.000,216.000
early_ctr,2191.0,0.003,0.000,0.074
early_avg_position,2191.0,14.924,0.336,102.888
early_active_days,2191.0,13.719,2.000,15.000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


All five features are numeric (the slice has no categorical features), and they all come from 1–15 March, so they exist at the decision moment, the end of 15 March. Missing values are filled with 0, which is only used if a count above shows a missing value.

| Feature | Meaning | Missing handling | Available before the decision moment? |
|---|---|---|---|
| early_impressions | total search impressions on days 1–15 | none missing after the 20-impression filter | Yes |
| early_clicks | total clicks on days 1–15 | filled with 0 if missing | Yes |
| early_ctr | early clicks divided by early impressions | none missing after the filter | Yes |
| early_avg_position | mean of the daily average position | filled with 0 if missing | Yes |
| early_active_days | days in 1–15 with impressions above 0 | none missing | Yes |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

after_splits = list(GroupKFold(n_splits=5).split(X, y, groups))

def grouped_auc(cols, target):
    Xc = df[cols].fillna(0)
    p = np.zeros(len(df))
    for tr, te in after_splits:
        m = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=0, n_jobs=-1)
        m.fit(Xc.iloc[tr], target.iloc[tr])
        p[te] = m.predict_proba(Xc.iloc[te])[:, 1]
    return roc_auc_score(target, p)

# 1) name check: no feature comes from the label window or a product flag
print("suspicious feature names:", [c for c in features if any(w in c for w in ("late", "trend", "label", "declin"))])

# 2) the honest score, a deliberately leaky column, and shuffled labels
print("honest AUC:", round(grouped_auc(features, y), 3))
df["late_vs_early"] = (df["late_impressions"] / 16) / (df["early_impressions"] / 15)   # built from the label window, on purpose
print("AUC with a leaky column:", round(grouped_auc(features + ["late_vs_early"], y), 3))
df = df.drop(columns=["late_vs_early"])
y_shuffled = pd.Series(np.random.default_rng(0).permutation(y.values), index=y.index)
print("AUC with shuffled labels (should be near 0.5):", round(grouped_auc(features, y_shuffled), 3))

# 3) privacy check: identifiers are hashes and the frame has no url or query columns
print("identifier example (shortened):", df["client_hash_id"].iloc[0][:12], "...")
print("url/query columns:", [c for c in frame.columns if "url" in c.lower() or "query" in c.lower()])

suspicious feature names: []
honest AUC: 0.555
AUC with a leaky column: 1.0
AUC with shuffled labels (should be near 0.5): 0.487
identifier example (shortened): client_0797f ...
url/query columns: []


Leakage hunt (measured, grouped by client): no feature name points to the label window or a product flag. The honest AUC is about 0.55. Adding a column built from 16–31 March pushes AUC to about 1.0, the fingerprint of leakage, and I removed it. With shuffled labels AUC is close to 0.5, so the pipeline does not create signal by itself. Privacy: identifiers are hashes, and the slice has no URL or query columns. One built-in link: the label compares the late rate with 0.8 times the early rate, so early impressions appear in both the label rule and the features. That is not future information, but it means high-volume pages are more likely to fall back (regression to the mean).

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


- **late_impressions and label_declining:** they come from 16–31 March, the label window, so using them as inputs would be leakage.
- **trend_direction, trend_pct, is_declining_label (starter CSV):** they encode the decline label or a version of it.
- **All GA4 columns:** present for only about 4% of rows, and a missing value there is not zero engagement.
- **client_hash_id and content_hash_id:** identifiers, used only to group rows and split by client, never as features.
- **June 2026:** kept sealed as a test month and never used to build the label.
- **fact_content_query_90d:** not used in this slice; its fixed 90-day window could overlap the label period.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.